<a href="https://colab.research.google.com/github/pressleydavid/hw5/blob/main/HW6_ST554_DavidPressley.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW6
**David Pressley**

**Course:** ST554

**Due:** March 3, 2026  

**Submission:** Save to GitHub repo, submit repo link on Moodle

The purpose of this homework is to get a little more practice with SQL and gain some practice with creating classes.

**Instructions:**
- Include markdown text describing what you are doing, even when not explicitly asked for
- Add comments to .py file explaining work
- Save updates to GitHub as you work through it
- No edits after the due date
- check sharing settings
- use hw5 repo. Develop on main.

In [3]:
# import ability to use Markdown in Code Cell displays
from IPython.display import display, Markdown

## Part I - More Practice Querying a Database (16 pts)
The purpose of this homework is to get a little more practice with SQL and gain some practice with creating classes.

### Question 1
Connect to the database and then look at all of the tables in the database (use `read_sql()` from `pandas` to have this returned as a data frame). (2 pts)

In [8]:
import pandas as pd
import sqlite3
import urllib.request

url = "https://github.com/jknecht/baseball-archive-sqlite/releases/download/2022/lahman_1871-2022.sqlite"
output_file = "lahman.sqlite"

urllib.request.urlretrieve(url, output_file)

con = sqlite3.connect('lahman.sqlite')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", con)
display(Markdown(f"### Tables in the Lahman Database"))
print(tables)


### Tables in the Lahman Database

                   name
0           AllstarFull
1           Appearances
2        AwardsManagers
3         AwardsPlayers
4   AwardsShareManagers
5    AwardsSharePlayers
6               Batting
7           BattingPost
8        CollegePlaying
9              Fielding
10           FieldingOF
11      FieldingOFsplit
12         FieldingPost
13           HallOfFame
14            HomeGames
15             Managers
16         ManagersHalf
17                Parks
18               People
19             Pitching
20         PitchingPost
21             Salaries
22              Schools
23           SeriesPost
24                Teams
25      TeamsFranchises
26            TeamsHalf


### Question 2
Using SQL, construct a table of hall of fame pitchers (any hall of famer that pitched) that gives the `playerID` and their total (sum) for `GS`, `G`, `W`, `L`, `IPOuts`, `CG`, `SHO`, and `SV` columns. The summing can be done in `pandas` or in the SQL call. (6 pts)

In [101]:
# write out table information for quick access to schema
pragma_pitchers = pd.read_sql("PRAGMA table_info(Pitching)", con)
pragma_pitchers = pragma_pitchers.sort_values("name")
display(Markdown(f"### Description of the Pitching Table"))
print(pragma_pitchers)

pragma_hof = pd.read_sql("PRAGMA table_info(HallOfFame)", con)
pragma_hof = pragma_hof.sort_values("name")
display(Markdown(f"### Description of the Hall of Fame Table"))
print(pragma_hof)

# how many unique players in HoF were inducted (pitchers inducted should be less than or equal to this number)
hof_check_unique = pd.read_sql("SELECT count(DISTINCT playerID), inducted FROM HallofFame WHERE inducted = 'Y'", con)
hof_check_unique

# group pitcher record count by playerID
pitcher_record_count = pd.read_sql("select playerID, count(playerID) as entry_count from pitching group by playerID having count(playerID) > 1", con)
pitcher_record_count


# LHS is HallofFame where inducted
# RHS is Pitchers
# inner join (default join) to get pitchers inducted
# BUGFIX: need to 'flatten' hall of fame data first to account for multiple inductions, then sum. Use subquery in from statement to get distinct playerID
hof_pitchers = pd.read_sql("""
    SELECT
        l.playerID,
        l.inducted,
        count(l.playerID) as entry_count,
        SUM(r.GS) as total_games_started,
        SUM(r.G) as total_games,
        SUM(r.W) as total_wins,
        SUM(r.L) as total_losses,
        SUM(r.IPOuts) as total_outs_pitched,
        SUM(r.CG) as total_complete_games,
        SUM(r.SHO) as total_shutouts,
        SUM(r.SV) as total_saves
    FROM (
        SELECT DISTINCT playerID, inducted
        FROM HallofFame
        WHERE inducted = 'Y'
    ) as l
    INNER JOIN
    Pitching as r
    ON l.playerID = r.playerID
    GROUP BY l.playerID
    HAVING count(l.playerID) > 1;
    """, con)
display(Markdown(f"### Selected totals for pitchers inducted into Hall of Fame"))
hof_pitchers


check_anson = pd.read_sql("""
    SELECT playerID, yearID, votedBy, category, inducted
    FROM HallofFame
    WHERE playerID = 'ansonca01' AND inducted = 'Y'
""", con)
print(check_anson)

query_combined = """
SELECT l.playerID, l.inducted, l.yearID, r.W, r.G
FROM HallofFame l
JOIN Pitching r ON l.playerID = r.playerID
WHERE l.inducted = 'Y' AND l.playerID = 'ansonca01'
"""

df = pd.read_sql(query_combined, con)
df



# sum_pitchers = hof_pitchers.read_sql("""select sum(W) from  )

    playerID  yearid     votedBy category inducted
0  ansonca01    1939  Old Timers   Player        Y


,playerID,inducted,yearid,W,G
0,ansonca01,Y,1939,0,2
1,ansonca01,Y,1939,0,1


### Question 3
For all of the hall of fame pitchers, use SQL to create a table of their batting statistics. Namely, the `playerID` and their total (sum) for `AB`, `R`, `H`, `HR`, `RBI`, `BB`, and `SO`. The summing can be done in `pandas` or in the SQL call. (4 pts)

### Question 4
Using `pandas` join the previous two tables together by pitcher. (If you want, try to do all of this via SQL! Not required though, feel free to use `pd.merge()` if you'd like) (4 pts)

## Part II - Messing with Classes

### Question 5
Create a class called `SLR_slope_simulator` that encapsulates the simulation of the sampling distribution of the slope estimator.

Recall we assume the following model for SLR:

$$Y_i = \beta_0 + \beta_1 x_i + E_i$$

where the $E_i$ are assumed to be independent and identically distributed from a Normal distribution with mean 0 and variance $\sigma^2$.

#### Class Definition

Define the `SLR_slope_simulator` class with:
- `__init__`: arguments `self`, `beta_0`, `beta_1`, `x`, `sigma`, and `seed`. Create initial attributes of `beta_0`, `beta_1`, `sigma`, `x`, `n`, `rng`, and `slopes` (an empty list).
- `generate_data`: generates one dataset (returning x and y)
- `fit_slope`: takes in x and y, fits SLR model, returns estimated slope
- `run_simulations`: takes number of simulations, uses `generate_data()` and `fit_slope()` in a loop, modifies `slopes` attribute
- `plot_sampling_distribution`: checks if slopes has length > 0, plots histogram if so
- `find_prob`: takes a `value` and `sided` argument ("above", "below", or "two-sided") and approximates the corresponding probability

#### Testing the Class

Create an instance of the object with `beta_0 = 12`, `beta_1 = 2`, `x = np.array(list(np.linspace(start = 0, stop = 10, num = 11))*3)`, `sigma = 1`, and `seed = 10`.

#### Call `run_simulations()` before running simulations (should return error message)

#### Run 10,000 simulations

#### Plot the sampling distribution

#### Approximate the two-sided probability of being larger than 2.1

#### Print out the simulated slopes